# Data Preparation: OULAD Feature Store

**Module**: WM9QG-15 Fundamentals of AI and Data Mining  
**CRISP-DM Phase**: Data Understanding & Data Preparation

---

## Overview

This notebook builds a comprehensive **feature store** for the Early Warning System (EWS) using the Open University Learning Analytics Dataset (OULAD). The feature store is a weekly snapshot of each student's state, enabling time-series aware predictions.

### Output
- `ews_feature_store.csv`: Weekly student snapshots with ~80+ features

### Feature Categories
1. **Static Features**: Demographics, course info, registration timing
2. **Dynamic Features**: Cumulative VLE engagement by activity type
3. **Assessment Progress**: Weekly cumulative scores and weights
4. **Target Variables**: Pass/Fail classification and final score regression

## Table of Contents

1. [Setup & Configuration](#1-setup)
2. [Load OULAD Tables](#2-load-data)
3. [Static Features](#3-static-features)
   - 3.1 Demographics
   - 3.2 Course Features
   - 3.3 Registration Features
4. [Target Variables](#4-targets)
   - 4.1 Classification Target (Pass/Fail)
   - 4.2 Regression Target (Final Score)
5. [VLE Engagement Features](#5-vle-features)
   - 5.1 Enrich VLE Data with Activity Types
   - 5.2 Weekly Aggregation by Activity Type
   - 5.3 Pivot and Cumulative Features
6. [Weekly Assessment Progress](#6-assessment-progress)
   - 6.1 Merge Assessment Metadata
   - 6.2 Cumulative Weekly Scores
   - 6.3 "Mathematically Impossible to Pass" Flag
7. [Build Final Feature Store](#7-build-store)
8. [Export & Validation](#8-export)

---
## 1. Setup & Configuration <a id="1-setup"></a>

In [1]:
%reset -f

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# Directory configuration
DATA_DIR = Path("..") / "data" / "raw"
PROCESSED_DIR = Path("..") / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Consistent key columns used throughout
KEY_COLS = ["student_id", "code_module", "code_presentation"]

print("Configuration:")
print(f"  Raw data directory: {DATA_DIR.resolve()}")
print(f"  Output directory: {PROCESSED_DIR.resolve()}")

Configuration:
  Raw data directory: /Users/wasimhoque/Documents/Data Mining/WM9QG-Group-Project/data/raw
  Output directory: /Users/wasimhoque/Documents/Data Mining/WM9QG-Group-Project/data/processed


---
## 2. Load OULAD Tables <a id="2-load-data"></a>

The OULAD dataset consists of 7 interconnected tables:

| Table | Description | Rows |
|-------|-------------|------|
| `courses` | Module presentations and their lengths | 22 |
| `assessments` | Assessment metadata (type, weight, date) | 206 |
| `vle` | VLE resource metadata | 6,364 |
| `studentInfo` | Student demographics and final results | 32,593 |
| `studentRegistration` | Registration/unregistration dates | 32,593 |
| `studentAssessment` | Student assessment scores | 173,912 |
| `studentVle` | Daily VLE click interactions | 10,655,280 |

In [3]:
def load_csv(name: str) -> pd.DataFrame:
    """Load a CSV file from the raw data directory."""
    return pd.read_csv(DATA_DIR / name)

# Load all tables
assessments          = load_csv("assessments.csv")
courses              = load_csv("courses.csv")
student_assessment   = load_csv("studentAssessment.csv")
student_info         = load_csv("studentInfo.csv")
student_registration = load_csv("studentRegistration.csv")
student_vle          = load_csv("studentVle.csv")
vle                  = load_csv("vle.csv")

# Standardize column naming: id_student -> student_id
# This ensures consistency across all tables
student_assessment   = student_assessment.rename(columns={"id_student": "student_id"})
student_info         = student_info.rename(columns={"id_student": "student_id"})
student_registration = student_registration.rename(columns={"id_student": "student_id"})
student_vle          = student_vle.rename(columns={"id_student": "student_id"})

# Summary
tables = {
    "assessments": assessments,
    "courses": courses,
    "student_assessment": student_assessment,
    "student_info": student_info,
    "student_registration": student_registration,
    "student_vle": student_vle,
    "vle": vle,
}

print("Loaded OULAD tables:")
for name, df in tables.items():
    print(f"  {name}: {df.shape[0]:,} rows, {df.shape[1]} cols")

Loaded OULAD tables:
  assessments: 206 rows, 6 cols
  courses: 22 rows, 3 cols
  student_assessment: 173,912 rows, 5 cols
  student_info: 32,593 rows, 12 cols
  student_registration: 32,593 rows, 5 cols
  student_vle: 10,655,280 rows, 6 cols
  vle: 6,364 rows, 6 cols


---
## 3. Static Features <a id="3-static-features"></a>

Static features remain constant throughout the module presentation. These include student demographics, course metadata, and registration timing.

### 3.1 Demographics

Demographic features from `studentInfo`:
- **gender**: Student's gender (M/F)
- **region**: Geographic region during study
- **highest_education**: Educational background on entry
- **imd_band**: Index of Multiple Deprivation (socioeconomic indicator)
- **age_band**: Age range (0-35, 35-55, 55+)
- **num_of_prev_attempts**: Previous attempts at this module
- **studied_credits**: Total credits currently studying
- **disability**: Whether student has declared a disability

In [4]:
demographics = student_info[KEY_COLS + [
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "num_of_prev_attempts",
    "studied_credits",
    "disability",
]].copy()

print(f"Demographics shape: {demographics.shape}")
demographics.head()

Demographics shape: (32593, 11)


,student_id,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability
0,11391,AAA,2013J,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N
1,28400,AAA,2013J,F,Scotland,HE Qualification,20-30%,35-55,0,60,N
2,30268,AAA,2013J,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y
3,31604,AAA,2013J,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N
4,32885,AAA,2013J,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N


### 3.2 Course Features

Course-level features from the `courses` table:
- **module_presentation_length**: Duration of the module in days

In [5]:
course_features = courses[["code_module", "code_presentation", "module_presentation_length"]].copy()

print(f"Course features shape: {course_features.shape}")
course_features.head()

Course features shape: (22, 3)


,code_module,code_presentation,module_presentation_length
0,AAA,2013J,268
1,AAA,2014J,269
2,BBB,2013J,268
3,BBB,2014J,262
4,BBB,2013B,240


### 3.3 Registration Features

Registration timing from `studentRegistration`:
- **date_registration**: Days relative to module start (negative = before start)
- **date_unregistration**: Day of withdrawal (NaN if completed)

In [6]:
registration = student_registration[KEY_COLS + [
    "date_registration",
    "date_unregistration"
]].copy()

# Ensure numeric types
registration["date_registration"] = pd.to_numeric(
    registration["date_registration"], errors="coerce"
)
registration["date_unregistration"] = pd.to_numeric(
    registration["date_unregistration"], errors="coerce"
)

registration_features = registration.copy()

print(f"Registration features shape: {registration_features.shape}")
registration_features.head()

Registration features shape: (32593, 5)


,student_id,code_module,code_presentation,date_registration,date_unregistration
0,11391,AAA,2013J,-159.0,NaN
1,28400,AAA,2013J,-53.0,NaN
2,30268,AAA,2013J,-92.0,12.0
3,31604,AAA,2013J,-52.0,NaN
4,32885,AAA,2013J,-176.0,NaN


---
## 4. Target Variables <a id="4-targets"></a>

We create two target variables:
1. **Classification**: Binary pass/fail prediction
2. **Regression**: Final weighted score prediction

### 4.1 Classification Target: Pass vs Fail/Withdraw

The `final_result` column has four possible values:
- **Pass** / **Distinction** â†’ `target_pass = 1`
- **Fail** / **Withdrawn** â†’ `target_pass = 0`

In [7]:
targets_class = student_info[KEY_COLS + ["final_result"]].copy()
print(targets_class.head())
targets_class["target_pass"] = targets_class["final_result"].isin(["Pass", "Distinction"]).astype(int)
print(targets_class["target_pass"])

print("Classification target distribution:")
print(targets_class["final_result"].value_counts())
print(f"\ntarget_pass: {targets_class['target_pass'].value_counts().to_dict()}")

   student_id code_module code_presentation final_result
0       11391         AAA             2013J         Pass
1       28400         AAA             2013J         Pass
2       30268         AAA             2013J    Withdrawn
3       31604         AAA             2013J         Pass
4       32885         AAA             2013J         Pass
0        1
1        1
2        0
3        1
4        1
        ..
32588    0
32589    1
32590    1
32591    0
32592    1
Name: target_pass, Length: 32593, dtype: int64
Classification target distribution:
final_result
Pass           12361
Withdrawn      10156
Fail            7052
Distinction     3024
Name: count, dtype: int64

target_pass: {0: 17208, 1: 15385}


### 4.2 Regression Target: Final Weighted Score

The final score is computed as the weighted sum of all assessment scores:

$$\text{target\_score} = \sum_{i} \text{score}_i \times \frac{\text{weight}_i}{100}$$

We also compute:
- **target_score_norm**: Normalized score (performance on submitted work only)
- **weight_covered**: Total weight of completed assessments

In [8]:
# Join student scores to assessment weights + module keys
sa = student_assessment.merge(
    assessments[["id_assessment", "code_module", "code_presentation", "weight"]],
    on="id_assessment",
    how="left",
)

# Ensure numeric types
sa["score"] = pd.to_numeric(sa["score"], errors="coerce")
sa["weight"] = pd.to_numeric(sa["weight"], errors="coerce")

# Drop rows lacking weight/score
sa = sa.dropna(subset=["weight", "score", "code_module", "code_presentation"])

# Calculate weighted contribution (0-100 scale)
sa["weighted_contrib"] = sa["score"] * (sa["weight"] / 100.0)

# Aggregate per student
targets_reg = (
    sa.groupby(KEY_COLS, as_index=False)
      .agg(
          target_score=("weighted_contrib", "sum"),
          weight_covered=("weight", "sum"),
      )
)

# Normalized score: performance on submitted work only
targets_reg["target_score_norm"] = np.where(
    targets_reg["weight_covered"] > 0,
    targets_reg["target_score"] * (100.0 / targets_reg["weight_covered"]),
    np.nan
)

# Merge classification and regression targets
targets = targets_class.merge(
    targets_reg[KEY_COLS + ["target_score", "target_score_norm", "weight_covered"]],
    on=KEY_COLS,
    how="left"
)

print(f"Targets shape: {targets.shape}")
print(f"Missing target_score: {targets['target_score'].isna().sum():,}")
targets.head()

Targets shape: (32593, 8)
Missing target_score: 6,773


,student_id,code_module,code_presentation,final_result,target_pass,target_score,target_score_norm,weight_covered
0,11391,AAA,2013J,Pass,1,82.4,82.4,100.0
1,28400,AAA,2013J,Pass,1,65.4,65.4,100.0
2,30268,AAA,2013J,Withdrawn,0,NaN,NaN,NaN
3,31604,AAA,2013J,Pass,1,76.3,76.3,100.0
4,32885,AAA,2013J,Pass,1,55.0,55.0,100.0


---
## 5. VLE Engagement Features <a id="5-vle-features"></a>

VLE (Virtual Learning Environment) engagement features capture student interactions with online course materials. We create cumulative features by activity type to track engagement patterns over time.

### 5.1 Enrich VLE Data with Activity Types

Join `studentVle` with `vle` to get the activity type for each interaction. Activity types include:
- `forumng`: Forum discussions
- `oucontent`: Course content pages
- `resource`: Downloadable resources
- `quiz`: Online quizzes
- And many more...

In [9]:
# Enrich VLE data with activity_type
vle_enriched = student_vle.merge(
    vle[["id_site", "code_module", "code_presentation", "activity_type"]],
    on=["id_site", "code_module", "code_presentation"],
    how="left",
)

# Normalize activity_type for stable column names
vle_enriched["activity_type"] = (
    vle_enriched["activity_type"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
    .str.lower()
)

# Convert date (days since start) to week number
vle_enriched["week"] = (pd.to_numeric(vle_enriched["date"], errors="coerce") // 7).astype("Int64")

print(f"VLE enriched shape: {vle_enriched.shape}")
print(f"\nActivity types: {vle_enriched['activity_type'].nunique()}")
print(vle_enriched["activity_type"].value_counts())

VLE enriched shape: (10655280, 8)

Activity types: 20
activity_type
forumng           2408457
oucontent         1963782
subpage           1949898
homepage          1735226
quiz               914573
resource           813185
url                407716
ouwiki             178165
oucollaborate       65793
externalquiz        50910
page                47549
questionnaire       44905
ouelluminate        22360
glossary            17435
dataplus            13506
dualpane            10153
htmlactivity         6835
folder               4678
sharedsubpage         148
repeatactivity          6
Name: count, dtype: int64


### 5.2 Weekly Aggregation by Activity Type

For each student-week-activity combination, compute:
- **weekly_clicks**: Total clicks on that activity type
- **weekly_active_days**: Number of distinct days with activity
- **weekly_n_resources**: Number of unique resources accessed

In [10]:
weekly_by_type = (
    vle_enriched
    .groupby(KEY_COLS + ["week", "activity_type"], as_index=False)
    .agg(
        weekly_clicks=("sum_click", "sum"),
        weekly_active_days=("date", "nunique"),
        weekly_n_resources=("id_site", "nunique"),
    )
)

print(f"Weekly by type shape: {weekly_by_type.shape}")
weekly_by_type.head()

Weekly by type shape: (2518170, 8)


,student_id,code_module,code_presentation,week,activity_type,weekly_clicks,weekly_active_days,weekly_n_resources
0,6516,AAA,2014J,-4,forumng,33,1,1
1,6516,AAA,2014J,-4,homepage,16,2,1
2,6516,AAA,2014J,-4,oucontent,57,2,5
3,6516,AAA,2014J,-4,resource,2,1,1
4,6516,AAA,2014J,-4,subpage,2,1,1


### 5.3 Pivot and Cumulative Features

Transform the long-format data into wide format with one column per activity type, then compute cumulative sums. This creates features like:
- `cum_weekly_clicks_forumng`: Cumulative forum clicks
- `cum_weekly_clicks_oucontent`: Cumulative content page views
- etc.

In [11]:
def pivot_metric(df: pd.DataFrame, value_col: str, prefix: str) -> pd.DataFrame:
    """
    Pivot a metric from long to wide format by activity_type.
    
    Args:
        df: DataFrame with KEY_COLS + [week, activity_type, value_col]
        value_col: Column to pivot
        prefix: Prefix for new column names
    
    Returns:
        Wide-format DataFrame with one column per activity type
    """
    wide = (
        df.pivot_table(
            index=KEY_COLS + ["week"],
            columns="activity_type",
            values=value_col,
            aggfunc="sum",
            fill_value=0,
        )
        .reset_index()
    )
    # Rename columns: activity_type -> prefix_activity_type
    wide.columns = [
        f"{prefix}_{c}".lower().replace(" ", "_")
        if c not in KEY_COLS + ["week"] else c
        for c in wide.columns
    ]
    return wide

# Pivot each metric
weekly_clicks_wide = pivot_metric(weekly_by_type, "weekly_clicks", "weekly_clicks")
weekly_days_wide   = pivot_metric(weekly_by_type, "weekly_active_days", "weekly_days")
weekly_res_wide    = pivot_metric(weekly_by_type, "weekly_n_resources", "weekly_resources")

# Merge all wide metrics
weekly_wide = (
    weekly_clicks_wide
    .merge(weekly_days_wide, on=KEY_COLS + ["week"], how="left")
    .merge(weekly_res_wide, on=KEY_COLS + ["week"], how="left")
)

# Compute cumulative features
weekly_wide = weekly_wide.sort_values(KEY_COLS + ["week"]).copy()
weekly_feature_cols = [c for c in weekly_wide.columns if c not in KEY_COLS + ["week"]]
cum_cols = ["cum_" + c for c in weekly_feature_cols]

weekly_wide[cum_cols] = (
    weekly_wide.groupby(KEY_COLS)[weekly_feature_cols]
              .cumsum()
              .rename(columns={c: "cum_" + c for c in weekly_feature_cols})
)

# Keep only cumulative features (these are what we use for modeling)
ews_features = weekly_wide[KEY_COLS + ["week"] + cum_cols].copy()

print(f"EWS features shape: {ews_features.shape}")
print(f"Number of cumulative features: {len(cum_cols)}")
ews_features.head()

EWS features shape: (627031, 64)
Number of cumulative features: 60


,student_id,code_module,code_presentation,week,cum_weekly_clicks_dataplus,cum_weekly_clicks_dualpane,cum_weekly_clicks_externalquiz,cum_weekly_clicks_folder,cum_weekly_clicks_forumng,cum_weekly_clicks_glossary,...,cum_weekly_resources_ouelluminate,cum_weekly_resources_ouwiki,cum_weekly_resources_page,cum_weekly_resources_questionnaire,cum_weekly_resources_quiz,cum_weekly_resources_repeatactivity,cum_weekly_resources_resource,cum_weekly_resources_sharedsubpage,cum_weekly_resources_subpage,cum_weekly_resources_url
0,6516,AAA,2014J,-4,0,0,0,0,33,0,...,0,0,0,0,0,0,1,0,1,0
1,6516,AAA,2014J,-3,0,0,0,0,46,0,...,0,0,0,0,0,0,5,0,4,1
2,6516,AAA,2014J,-2,0,0,0,0,47,0,...,0,0,0,0,0,0,5,0,4,1
3,6516,AAA,2014J,-1,0,0,0,0,64,0,...,0,0,0,0,0,0,7,0,7,2
4,6516,AAA,2014J,0,0,0,0,0,124,0,...,0,0,0,0,0,0,8,0,9,4


### 5.4 Extra EWS features: totals / deltas / trends / breadth

In [12]:
# --- 1) Identify cumulative column groups ---
cum_click_cols = [c for c in ews_features.columns if c.startswith("cum_weekly_clicks_")]
cum_days_cols  = [c for c in ews_features.columns if c.startswith("cum_weekly_days_")]
cum_res_cols   = [c for c in ews_features.columns if c.startswith("cum_weekly_resources_")]

# --- 2) Aggregate totals (overall engagement so far) ---
ews_features["cum_clicks_total"] = ews_features[cum_click_cols].sum(axis=1)
ews_features["cum_days_total"]   = ews_features[cum_days_cols].sum(axis=1)
ews_features["cum_res_total"]    = ews_features[cum_res_cols].sum(axis=1)

# Sort to ensure diffs are correct per student-module-presentation
ews_features = ews_features.sort_values(KEY_COLS + ["week"]).copy()

# --- 3) Delta features (what happened since last week = recency) ---
ews_features["delta_clicks_total"] = (
    ews_features.groupby(KEY_COLS)["cum_clicks_total"].diff().fillna(0)
)
ews_features["delta_days_total"] = (
    ews_features.groupby(KEY_COLS)["cum_days_total"].diff().fillna(0)
)
ews_features["delta_res_total"] = (
    ews_features.groupby(KEY_COLS)["cum_res_total"].diff().fillna(0)
)

# --- 4) Trend / normalised engagement (fair across weeks) ---
eps = 1e-6
ews_features["avg_clicks_per_week"] = ews_features["cum_clicks_total"] / (ews_features["week"] + 1 + eps)
ews_features["avg_days_per_week"]   = ews_features["cum_days_total"]   / (ews_features["week"] + 1 + eps)

# --- 5) Breadth / diversity of engagement ---
# number of activity types ever used so far (based on clicks)
ews_features["n_active_activity_types"] = (ews_features[cum_click_cols] > 0).sum(axis=1)

# --- 6) Optional simple flags ---
ews_features["no_activity_last_week"]  = (ews_features["delta_clicks_total"] == 0).astype(int)
ews_features["low_activity_last_week"] = (ews_features["delta_clicks_total"] < 5).astype(int)

print("✓ Added delta / trend / breadth features")
print("New columns added:",
      ["cum_clicks_total","cum_days_total","cum_res_total",
       "delta_clicks_total","delta_days_total","delta_res_total",
       "avg_clicks_per_week","avg_days_per_week",
       "n_active_activity_types","no_activity_last_week","low_activity_last_week"])

✓ Added delta / trend / breadth features
New columns added: ['cum_clicks_total', 'cum_days_total', 'cum_res_total', 'delta_clicks_total', 'delta_days_total', 'delta_res_total', 'avg_clicks_per_week', 'avg_days_per_week', 'n_active_activity_types', 'no_activity_last_week', 'low_activity_last_week']


---
## 6. Weekly Assessment Progress <a id="6-assessment-progress"></a>

Track cumulative assessment performance week-by-week. This enables the model to know, at any given week, how well a student is doing academically.

**Key Features:**
- `cumulative_weighted_score_assess`: Running total of weighted assessment scores
- `weight_covered_assess`: Running total of completed assessment weights
- `mathematically_impossible_to_pass`: Flag when passing is no longer achievable

### 6.1 Merge Assessment Metadata

Join student assessment scores with assessment metadata (date, weight, type). We use the **scheduled assessment date** (not submission date) to avoid data leakage.

In [13]:
# Ensure numeric types
assessments["id_assessment"] = pd.to_numeric(assessments["id_assessment"], errors="coerce")
assessments["date"] = pd.to_numeric(assessments["date"], errors="coerce")
assessments["weight"] = pd.to_numeric(assessments["weight"], errors="coerce")

student_assessment["id_assessment"] = pd.to_numeric(student_assessment["id_assessment"], errors="coerce")
student_assessment["score"] = pd.to_numeric(student_assessment["score"], errors="coerce")

# Merge assessment metadata
sa_weekly = student_assessment.merge(
    assessments[["code_module", "code_presentation", "id_assessment", "assessment_type", "date", "weight"]],
    on="id_assessment",
    how="left",
    validate="many_to_one"
)

print(f"Missing weights after merge: {sa_weekly['weight'].isna().sum()}")
print(f"Missing assessment dates after merge: {sa_weekly['date'].isna().sum()}")

# Drop rows with missing dates (some exams don't have scheduled dates)
sa_weekly = sa_weekly.dropna(subset=["date"]).copy()

Missing weights after merge: 0
Missing assessment dates after merge: 2865


### 6.2 Define Assessment Week and Handle Multiple Attempts

Convert assessment dates to week numbers. For students with multiple attempts on the same assessment, we take the **best score** (simulating resit policy).

In [14]:
# Convert date to week (using scheduled assessment date, not submission date)
sa_weekly["week"] = (sa_weekly["date"] // 7).astype("Int64")
sa_weekly = sa_weekly[sa_weekly["week"] >= 0].copy()

print(f"Assessment weeks: {sa_weekly['week'].min()} â†’ {sa_weekly['week'].max()}")

# Resolve multiple attempts: take best score
sa_best = (
    sa_weekly.groupby(["student_id", "code_module", "code_presentation", "id_assessment"], as_index=False)
      .agg(
          week=("week", "first"),
          weight=("weight", "first"),
          assessment_type=("assessment_type", "first"),
          score=("score", "max")  # Best score after resit
      )
)

print(f"\nAssessment records (after dedup): {sa_best.shape[0]:,}")
sa_best.head()

Assessment weeks: 1 â†’ 37

Assessment records (after dedup): 171,047


,student_id,code_module,code_presentation,id_assessment,week,weight,assessment_type,score
0,6516,AAA,2014J,1758,2,10.0,TMA,60.0
1,6516,AAA,2014J,1759,7,20.0,TMA,48.0
2,6516,AAA,2014J,1760,16,20.0,TMA,63.0
3,6516,AAA,2014J,1761,23,20.0,TMA,61.0
4,6516,AAA,2014J,1762,30,30.0,TMA,77.0


### 6.3 Cumulative Weekly Assessment Progress

For each student-week, compute:
- **weighted_score_week**: Score contribution from assessments due that week
- **weight_covered_week**: Assessment weight due that week
- **cumulative_weighted_score_assess**: Running total of all scores
- **weight_covered_assess**: Running total of completed weight

In [15]:
# Calculate weighted score contribution (score/100 * weight)
sa_best["weighted_score"] = (sa_best["score"] / 100.0) * sa_best["weight"]

# Aggregate by student-week
weekly_progress = (
    sa_best.groupby(["student_id", "code_module", "code_presentation", "week"], as_index=False)
           .agg(
               weighted_score_week=("weighted_score", "sum"),
               weight_covered_week=("weight", "sum")
           )
           .sort_values(["student_id", "code_module", "code_presentation", "week"])
)

# Compute cumulative sums
weekly_progress["cumulative_weighted_score_assess"] = (
    weekly_progress.groupby(["student_id", "code_module", "code_presentation"])["weighted_score_week"].cumsum()
)

weekly_progress["weight_covered_assess"] = (
    weekly_progress.groupby(["student_id", "code_module", "code_presentation"])["weight_covered_week"].cumsum()
)

print(f"Weekly progress shape: {weekly_progress.shape}")
weekly_progress.head(10)

Weekly progress shape: (123512, 8)


,student_id,code_module,code_presentation,week,weighted_score_week,weight_covered_week,cumulative_weighted_score_assess,weight_covered_assess
0,6516,AAA,2014J,2,6.000,10.0,6.000,10.0
1,6516,AAA,2014J,7,9.600,20.0,15.600,30.0
2,6516,AAA,2014J,16,12.600,20.0,28.200,50.0
3,6516,AAA,2014J,23,12.200,20.0,40.400,70.0
4,6516,AAA,2014J,30,23.100,30.0,63.500,100.0
5,8462,DDD,2013J,3,9.300,10.0,9.300,10.0
6,8462,DDD,2013J,7,10.375,12.5,19.675,22.5
7,8462,DDD,2013J,12,15.225,17.5,34.900,40.0
8,8462,DDD,2014J,2,4.650,5.0,4.650,5.0
9,8462,DDD,2014J,5,8.300,10.0,12.950,15.0


---
## 7. Build Final Feature Store <a id="7-build-store"></a>

Merge all features into a single feature store with one row per student-module-presentation-week.

In [16]:
# Start with VLE engagement features (this defines the week grid)
feature_store = (
    ews_features
    .merge(demographics, on=KEY_COLS, how="left")
    .merge(course_features, on=["code_module", "code_presentation"], how="left")
    .merge(registration_features, on=KEY_COLS, how="left")
    .merge(targets, on=KEY_COLS, how="left")
)

print(f"Feature store shape (before assessment progress): {feature_store.shape}")

Feature store shape (before assessment progress): (627031, 91)


### 7.1 Add Weekly Assessment Progress

Merge the cumulative assessment progress and forward-fill missing weeks.

In [17]:
# Filter to post-start weeks only (week >= 0)
feature_store = feature_store[feature_store["week"] >= 0].copy()

# Merge weekly assessment progress
feature_store = feature_store.merge(
    weekly_progress[["student_id", "code_module", "code_presentation", "week",
                     "weighted_score_week", "weight_covered_week",
                     "cumulative_weighted_score_assess", "weight_covered_assess"]],
    on=["student_id", "code_module", "code_presentation", "week"],
    how="left"
)

# Sort for proper forward-fill
feature_store = feature_store.sort_values(["student_id", "code_module", "code_presentation", "week"])

# Weekly increments: missing => 0 (no assessment that week)
feature_store["weighted_score_week"] = feature_store["weighted_score_week"].fillna(0.0)
feature_store["weight_covered_week"] = feature_store["weight_covered_week"].fillna(0.0)

# Cumulative values: forward-fill, then fill remaining with 0
feature_store["cumulative_weighted_score_assess"] = (
    feature_store.groupby(["student_id", "code_module", "code_presentation"])["cumulative_weighted_score_assess"]
          .ffill()
          .fillna(0.0)
)

feature_store["weight_covered_assess"] = (
    feature_store.groupby(["student_id", "code_module", "code_presentation"])["weight_covered_assess"]
          .ffill()
          .fillna(0.0)
)

print(f"Feature store shape (after assessment progress): {feature_store.shape}")

Feature store shape (after assessment progress): (579438, 95)


### 7.2 Add "Mathematically Impossible to Pass" Flag

A student is mathematically impossible to pass if their maximum possible final score (current score + remaining weight) is below 40.

In [18]:
# Get total weight per module-presentation
total_weight = (
    assessments.groupby(["code_module", "code_presentation"], as_index=False)["weight"]
               .sum()
               .rename(columns={"weight": "total_weight_assess"})
)

feature_store = feature_store.merge(total_weight, on=["code_module", "code_presentation"], how="left")

# Calculate max possible final score
PASS_MARK = 40

feature_store["remaining_weight_assess"] = feature_store["total_weight_assess"] - feature_store["weight_covered_assess"]
feature_store["max_possible_final_score_assess"] = feature_store["cumulative_weighted_score_assess"] + feature_store["remaining_weight_assess"]

# Flag: 1 if mathematically impossible to reach 40%
feature_store["mathematically_impossible_to_pass"] = (feature_store["max_possible_final_score_assess"] < PASS_MARK).astype(int)

print("Mathematically impossible to pass distribution:")
print(feature_store["mathematically_impossible_to_pass"].value_counts())

Mathematically impossible to pass distribution:
mathematically_impossible_to_pass
0    579438
Name: count, dtype: int64


---
## 8. Export & Validation <a id="8-export"></a>

In [19]:
# Export
out_path = PROCESSED_DIR / "ews_feature_store.csv"
feature_store.to_csv(out_path, index=False)
print(f"Saved feature store to: {out_path.resolve()}")

Saved feature store to: /Users/wasimhoque/Documents/Data Mining/WM9QG-Group-Project/data/processed/ews_feature_store.csv


In [20]:
# Final validation
print("="*60)
print("FEATURE STORE SUMMARY")
print("="*60)

print(f"\nShape: {feature_store.shape[0]:,} rows {feature_store.shape[1]} columns")
print(f"Weeks: {feature_store['week'].min()}  {feature_store['week'].max()}")

print("\nUnique counts:")
print(f"  Students: {feature_store['student_id'].nunique():,}")
print(f"  Modules: {feature_store['code_module'].nunique()}")
print(f"  Presentations: {feature_store['code_presentation'].nunique()}")

print("\nClass balance (target_pass):")
print(feature_store["target_pass"].value_counts(dropna=False))

print("\nMissing targets:")
print(f"  target_pass: {feature_store['target_pass'].isna().sum()}")
print(f"  target_score: {feature_store['target_score'].isna().sum():,}")

print("\nAssessment progress stats:")
print(f"  cumulative_weighted_score_assess: {feature_store['cumulative_weighted_score_assess'].describe().to_dict()}")
print(f"  weight_covered_assess: {feature_store['weight_covered_assess'].describe().to_dict()}")

print("\nFeature categories:")
cum_vle_cols = [c for c in feature_store.columns if c.startswith("cum_weekly")]
print(f"  Cumulative VLE features: {len(cum_vle_cols)}")
print(f"  Assessment progress features: 6")
print(f"  Demographic features: 8")
print(f"  Target variables: 4")

print("\nSample data:")
feature_store.head()

FEATURE STORE SUMMARY

Shape: 579,438 rows 99 columns
Weeks: 0  38

Unique counts:
  Students: 25,516
  Modules: 7
  Presentations: 4

Class balance (target_pass):
target_pass
1    435460
0    143978
Name: count, dtype: int64

Missing targets:
  target_pass: 0
  target_score: 7,155

Assessment progress stats:
  cumulative_weighted_score_assess: {'count': 579438.0, 'mean': 31.386964101422414, 'std': 28.345299302759276, 'min': 0.0, '25%': 6.5, '50%': 22.25, '75%': 53.040000000000006, 'max': 197.9}
  weight_covered_assess: {'count': 579438.0, 'mean': 41.104913726749025, 'std': 35.7939335301703, 'min': 0.0, '25%': 10.0, '50%': 30.0, '75%': 75.0, 'max': 200.0}

Feature categories:
  Cumulative VLE features: 60
  Assessment progress features: 6
  Demographic features: 8
  Target variables: 4

Sample data:


,student_id,code_module,code_presentation,week,cum_weekly_clicks_dataplus,cum_weekly_clicks_dualpane,cum_weekly_clicks_externalquiz,cum_weekly_clicks_folder,cum_weekly_clicks_forumng,cum_weekly_clicks_glossary,...,target_score_norm,weight_covered,weighted_score_week,weight_covered_week,cumulative_weighted_score_assess,weight_covered_assess,total_weight_assess,remaining_weight_assess,max_possible_final_score_assess,mathematically_impossible_to_pass
0,6516,AAA,2014J,0,0,0,0,0,124,0,...,63.5,100.0,0.0,0.0,0.0,0.0,200.0,200.0,200.0,0
1,6516,AAA,2014J,1,0,0,0,0,146,0,...,63.5,100.0,0.0,0.0,0.0,0.0,200.0,200.0,200.0,0
2,6516,AAA,2014J,2,0,0,0,0,158,0,...,63.5,100.0,6.0,10.0,6.0,10.0,200.0,190.0,196.0,0
3,6516,AAA,2014J,3,0,0,0,0,164,0,...,63.5,100.0,0.0,0.0,6.0,10.0,200.0,190.0,196.0,0
4,6516,AAA,2014J,4,7,0,0,0,184,0,...,63.5,100.0,0.0,0.0,6.0,10.0,200.0,190.0,196.0,0


---
## Summary

This notebook produced `ews_feature_store.csv` containing:

**Static Features:**
- Demographics: gender, region, education, IMD band, age, credits, disability
- Course: module presentation length
- Registration: registration date, unregistration date

**Dynamic VLE Features (60+ columns):**
- Cumulative clicks by activity type
- Cumulative active days by activity type
- Cumulative unique resources by activity type

**Assessment Progress Features:**
- `cumulative_weighted_score_assess`: Running academic performance
- `weight_covered_assess`: Running assessment completion
- `mathematically_impossible_to_pass`: Early detection flag

**Target Variables:**
- `target_pass`: Binary classification (1=Pass/Distinction, 0=Fail/Withdrawn)
- `target_score`: Final weighted score (regression)
- `target_score_norm`: Normalized score on submitted work

---